<a href="https://colab.research.google.com/github/OmaRecalde/machinelearning2026/blob/main/Test_parcial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad Práctico Experimental: Regresión Lineal Múltiple
**Asignatura:** Machine Learning 2 | **Docente:** Msc. Elvis Pachacama (c)
**Estudiante:** Omar Recalde | **Carrera:** Desarrollo de Software

## Objetivo
Construir un modelo de regresión lineal múltiple para estimar los gastos médicos (`charges`) a partir de las características de los asegurados.

## Pipeline del proyecto
Carga → EDA → Outliers → Preparación de variables → Escalado → Selección de atributos → Validación → Entrenamiento → Evaluación → Predicción

In [1]:
!pip -q install kagglehub

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Carga y comprensión de los datos
Se descarga el dataset directamente desde Kaggle con `kagglehub`, de forma automatizada y sin subir archivos manualmente.

In [2]:
ruta = kagglehub.dataset_download("mirichoi0218/insurance")
print("Ruta del dataset:", ruta)
print("Archivos:", os.listdir(ruta))

df = pd.read_csv(os.path.join(ruta, "insurance.csv"))
df.head()

Using Colab cache for faster access to the 'insurance' dataset.
Ruta del dataset: /kaggle/input/insurance
Archivos: ['insurance.csv']


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nInformación general:")
df.info()

Dimensiones: (1338, 7)

Tipos de datos:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [4]:
display(df.describe())
display(df.describe(include="object"))

for col in ["sex", "smoker", "region", "children"]:
    print(f"\n{col}:\n{df[col].value_counts()}")

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


,sex,smoker,region
count,1338,1338,1338
unique,2,2,4
top,male,no,southeast
freq,676,1064,364



sex:
sex
male      676
female    662
Name: count, dtype: int64

smoker:
smoker
no     1064
yes     274
Name: count, dtype: int64

region:
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64

children:
children
0    574
1    324
2    240
3    157
4     25
5     18
Name: count, dtype: int64


### Diccionario de variables
| Variable | Tipo | Descripción |
|---|---|---|
| age | Numérica | Edad del asegurado |
| sex | Categórica | Sexo (male/female) |
| bmi | Numérica | Índice de masa corporal |
| children | Numérica (discreta) | Número de dependientes |
| smoker | Categórica | Condición de fumador (yes/no) |
| region | Categórica | Región (northeast, northwest, southeast, southwest) |
| charges | Numérica | **Variable objetivo**: gastos médicos |

**Observación:** el dataset tiene 1338 registros y 7 columnas. Se espera una relación fuerte entre `smoker` y `charges`.